<a href="https://colab.research.google.com/github/Harshavardhan655/complier-design-/blob/main/CD_exp_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!apt-get update -qq
!apt-get install -y flex gcc


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:13.2.0-7ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison flex-doc
The following NEW packages will be installed:
  flex libfl-dev libfl2
0 upgraded, 3 newly installed, 0 to remove and 59 not upgraded.
Need to get 317 kB of archives.
After this operation, 1,068 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 flex amd64 2.6.4-8.2build1 [301 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libfl2 amd64 2.6.4-8.2build1 [9,768 B]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 libfl-dev amd64 2.6.4-8.2build1 [6,250 B

In [2]:

%%writefile octal.l
%{
#include <stdio.h>
#include <stdlib.h>
%}

%%
[0-7]+ {
    long long decimal = 0;

    for (int i = 0; yytext[i] != '\0'; i++) {
        decimal = decimal * 8 + (yytext[i] - '0');
    }

    printf("Octal number: %s\n", yytext);
    printf("Decimal equivalent: %lld\n", decimal);
}
[ \t\n]+  ;
[0-9]+ {
    printf("Invalid octal number: %s\n", yytext);
}
. {
    printf("Invalid character: %s\n", yytext);
}
%%

int main(void)
{
    printf("Enter an octal number: ");
    yylex();
    return 0;
}

int yywrap(void)
{
    return 1;
}


Writing octal.l


In [3]:

!flex octal.l
!gcc lex.yy.c -o octal -lfl
!echo "157" | ./octal


Enter an octal number: Octal number: 157
Decimal equivalent: 111


In [5]:

%%writefile analyzer.l
%{
#include <stdio.h>

int vowels = 0;
int consonants = 0;
int characters = 0;
int words = 0;
int lines = 0;
int last_was_newline = 1;
int any_character = 0;
%}

%%
[aeiouAEIOU] {
    vowels++;
    characters++;
    any_character = 1;
    last_was_newline = 0;
}

[a-zA-Z] {
    consonants++;
    characters++;
    any_character = 1;
    last_was_newline = 0;
}

[ \t\r\f\v]+ {
    characters += yyleng;
    any_character = 1;
    last_was_newline = 0;
}

\n {
    lines++;
    any_character = 1;
    last_was_newline = 1;
}

[^a-zA-ZaeiouAEIOU \t\r\f\v\n]+ {
    characters += yyleng;
    any_character = 1;
    last_was_newline = 0;
}

%%

int main(int argc, char *argv[])
{
    if (argc != 2) {
        printf("Usage: %s input.txt\n", argv[0]);
        return 1;
    }

    yyin = fopen(argv[1], "r");

    if (yyin == NULL) {
        perror("Error opening file");
        return 1;
    }

    /* Count words separately using whitespace boundaries. */
    int c;
    int in_word = 0;

    while ((c = fgetc(yyin)) != EOF) {
        if (c == ' ' || c == '\t' || c == '\n' ||
            c == '\r' || c == '\f' || c == '\v') {
            in_word = 0;
        } else if (!in_word) {
            words++;
            in_word = 1;
        }
    }

    rewind(yyin);

    yylex();

    if (any_character && !last_was_newline) {
        lines++;
    }

    printf("\n--- Text File Analysis ---\n");
    printf("Vowels      : %d\n", vowels);
    printf("Consonants  : %d\n", consonants);
    printf("Characters  : %d\n", characters);
    printf("Words       : %d\n", words);
    printf("Lines       : %d\n", lines);

    fclose(yyin);
    return 0;
}

int yywrap(void)
{
    return 1;
}


Writing analyzer.l
